In [1345]:
import pandas as pd
import re

In [1346]:
booking_requests_df = pd.read_csv("booking_requests.csv")
cleaners_df = pd.read_csv("cleaners.csv")
customers_df = pd.read_csv("customers.csv")
marketing_spend_df = pd.read_csv("marketing_spend.csv")

# 1. Customers

In [1347]:
customers_df.head()

,customer_id,signup_date,city,acquisition_channel
0,C000192,2025-01-01,Paris,Google Ads
1,C000039,2025-01-01,Paris,Google Ads
2,C000572,2025-01-01,Lyon,Organic
3,C000504,2025-01-01,Lyon,Meta Ads
4,C000307,2025-01-01,Paris,Google Ads


In [1348]:
customers_df.shape

(24662, 4)

In [1349]:
customers_df = customers_df.drop_duplicates()

In [1350]:
customers_df.info()
# ==> no missing values and no duplicates

<class 'pandas.DataFrame'>
RangeIndex: 24662 entries, 0 to 24661
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          24662 non-null  str  
 1   signup_date          24662 non-null  str  
 2   city                 24662 non-null  str  
 3   acquisition_channel  24662 non-null  str  
dtypes: str(4)
memory usage: 770.8 KB


In [1351]:
def detect_test(ch):
    if pd.isna(ch):
        return "Missing"
    elif re.search(r"tEst", str(ch), re.IGNORECASE):
        return "Test"
    else:
        return "Real"

In [1352]:
#Deleting Test accounts
customers_df["customer_real_account"] = customers_df["customer_id"].apply(detect_test)
customers_df

,customer_id,signup_date,city,acquisition_channel,customer_real_account
0,C000192,2025-01-01,Paris,Google Ads,Real
1,C000039,2025-01-01,Paris,Google Ads,Real
2,C000572,2025-01-01,Lyon,Organic,Real
3,C000504,2025-01-01,Lyon,Meta Ads,Real
4,C000307,2025-01-01,Paris,Google Ads,Real
...,...,...,...,...,...
24657,C_TEST25,2025-01-15,Paris,internal,Test
24658,C_TEST26,2025-01-15,Paris,internal,Test
24659,C_TEST27,2025-01-15,Paris,internal,Test
24660,C_TEST28,2025-01-15,Paris,internal,Test


In [1353]:
customers_df = customers_df[customers_df["customer_real_account"]=='Real']
customers_df = customers_df.drop('customer_real_account', axis=1)

In [1354]:
#check that the customer_id are unique and the cities
print('The number of unique customers : ',len(customers_df['customer_id'].unique()))
print('The cities are : ', list(customers_df['city'].unique()))

The number of unique customers :  24632
The cities are :  ['Paris', 'Lyon', 'Bordeaux', 'Lille']


In [1355]:
#check if the same customer_id had more than one signup_date
signup_counts = customers_df.groupby('customer_id')['signup_date'].count()
customers_df = customers_df[customers_df['customer_id'].isin(signup_counts[signup_counts == 1].index)]
customers_df.shape
# ==> each customer_id has only one sigup date

(24632, 4)

In [1356]:
#check if the same customer_id had more than one city
city_counts = customers_df.groupby('customer_id')['city'].count()
customers_df = customers_df[customers_df['customer_id'].isin(city_counts[city_counts == 1].index)]
customers_df.shape
# ==> each customer_id had only 

(24632, 4)

In [1357]:
#check the signup_date
dates = pd.to_datetime(customers_df['signup_date'], format='%Y-%m-%d', errors='coerce')
customers_df = customers_df[dates.notna()]
customers_df
# ==> dates are in good format

,customer_id,signup_date,city,acquisition_channel
0,C000192,2025-01-01,Paris,Google Ads
1,C000039,2025-01-01,Paris,Google Ads
2,C000572,2025-01-01,Lyon,Organic
3,C000504,2025-01-01,Lyon,Meta Ads
4,C000307,2025-01-01,Paris,Google Ads
...,...,...,...,...
24627,C024347,2026-06-30,Bordeaux,Meta Ads
24628,C022608,2026-06-30,Paris,Organic
24629,C024561,2026-06-30,Lille,Google Ads
24630,C024436,2026-06-30,Lille,Google Ads


In [1358]:
customers_df['year'] = pd.to_datetime(customers_df['signup_date']).dt.year
customers_df['month'] = pd.to_datetime(customers_df['signup_date']).dt.month
customers_df['day'] = pd.to_datetime(customers_df['signup_date']).dt.day
customers_df.head()

,customer_id,signup_date,city,acquisition_channel,year,month,day
0,C000192,2025-01-01,Paris,Google Ads,2025,1,1
1,C000039,2025-01-01,Paris,Google Ads,2025,1,1
2,C000572,2025-01-01,Lyon,Organic,2025,1,1
3,C000504,2025-01-01,Lyon,Meta Ads,2025,1,1
4,C000307,2025-01-01,Paris,Google Ads,2025,1,1


In [1359]:
customers_df['city'].unique()

<StringArray>
['Paris', 'Lyon', 'Bordeaux', 'Lille']
Length: 4, dtype: str

In [1360]:
customers_df['acquisition_channel'].unique()

<StringArray>
['Google Ads', 'Organic', 'Meta Ads', 'Referral']
Length: 4, dtype: str

In [1361]:
#analyse plus detaillee pour la table:
#compte le nb de customer par city
#compte le nb de customer par quarter et interprete la monotonie

# 2. cleaners

In [1362]:
cleaners_df.head()

,cleaner_id,application_date,city,acquisition_channel,onboarded_date,deactivated_date
0,P00027,2025-01-01,Paris,Meta Ads,2025-01-19,2025-01-30
1,P00002,2025-01-01,Paris,Meta Ads,2025-01-10,2025-12-31
2,P00028,2025-01-01,Paris,Job boards,2025-01-10,2025-08-24
3,P00014,2025-01-02,Paris,Meta Ads,2025-01-19,NaN
4,P00025,2025-01-04,Paris,Job boards,2025-01-23,2025-10-23


In [1363]:
cleaners_df.shape

(1511, 6)

In [1364]:
cleaners_df = cleaners_df.drop_duplicates()

In [1365]:
cleaners_df.info()
# ==> no duplicated rows

<class 'pandas.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   cleaner_id           1511 non-null   str  
 1   application_date     1511 non-null   str  
 2   city                 1511 non-null   str  
 3   acquisition_channel  1511 non-null   str  
 4   onboarded_date       1101 non-null   str  
 5   deactivated_date     537 non-null    str  
dtypes: str(6)
memory usage: 71.0 KB


In [1366]:
#Check if there are test accounts in cleaner_id
cleaners_df["cleaner_real_account"] = cleaners_df["cleaner_id"].apply(detect_test)
cleaners_df = cleaners_df[cleaners_df["cleaner_real_account"]=='Real']
cleaners_df = cleaners_df.drop('cleaner_real_account', axis=1)
cleaners_df.shape
# ==> no test accounts for cleaner

(1511, 6)

In [1367]:
cleaners_df["city"].unique()

<StringArray>
['Paris', 'Lyon', 'Bordeaux', 'Lille']
Length: 4, dtype: str

In [1368]:
# need to check the dates chronology

# 3. Booking_requests

In [1369]:
booking_requests_df.head()

,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating
0,R0000609,C000609,Lyon,2025-01-14,4,108.0,fulfilled,P00037,5.0
1,R0010556,C002995,Bordeaux,2025-05-07,3,81.0,fulfilled,P00186,4.0
2,R0001916,C001191,Lyon,2025-02-20,3,81.0,unfulfilled,NaN,NaN
3,R0061885,C009288,Paris,2026-01-14,3,81.0,fulfilled,P00212,3.0
4,R0015685,C000933,Paris,2025-07-12,3,81.0,fulfilled,P00136,4.0


In [1370]:
booking_requests_df['rating'].unique()

array([ 5.,  4., nan,  3.,  1.,  2.])

In [1371]:
booking_requests_df["status"].unique()

<StringArray>
['fulfilled', 'unfulfilled', 'cancelled_by_customer']
Length: 3, dtype: str

In [1372]:
booking_requests_df_init =booking_requests_df.shape

In [1373]:
booking_requests_df = booking_requests_df.drop_duplicates()
print("the number of duplicated booking requests:", booking_requests_df_init[0]-booking_requests_df.shape[0])

the number of duplicated booking requests: 1323


In [1374]:
booking_requests_df.info()

<class 'pandas.DataFrame'>
Index: 132384 entries, 0 to 133706
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   request_id    132384 non-null  str    
 1   customer_id   132384 non-null  str    
 2   city          132384 non-null  str    
 3   request_date  132384 non-null  str    
 4   hours         132384 non-null  int64  
 5   price_eur     131286 non-null  float64
 6   status        132384 non-null  str    
 7   cleaner_id    108912 non-null  str    
 8   rating        108912 non-null  float64
dtypes: float64(2), int64(1), str(6)
memory usage: 10.1 MB


In [1375]:
booking_requests_df_avant = booking_requests_df.copy()
booking_requests_df["booking_requests_df_rrequest_id"] = booking_requests_df["request_id"].apply(detect_test)
booking_requests_df = booking_requests_df[booking_requests_df["booking_requests_df_rrequest_id"]=='Real']
booking_requests_df.shape
print("The number of Test requests: ",booking_requests_df_avant.shape[0]-booking_requests_df.shape[0])
# ==> 220 Test Requests

The number of Test requests:  220


In [1376]:
booking_requests_df_avant = booking_requests_df.copy()
booking_requests_df["booking_requests_df_customer_id"] = booking_requests_df["customer_id"].apply(detect_test)
booking_requests_df = booking_requests_df[booking_requests_df["booking_requests_df_customer_id"]=='Real']
booking_requests_df.shape
print("The number of Test requests: ",booking_requests_df_avant.shape[0]-booking_requests_df.shape[0])

The number of Test requests:  0


In [1377]:
booking_requests_df_avant = booking_requests_df.copy()
booking_requests_df["booking_requests_df_cleaner_id"] = booking_requests_df["cleaner_id"].apply(detect_test)
booking_requests_df = booking_requests_df[booking_requests_df["booking_requests_df_cleaner_id"]=='Real']
booking_requests_df.shape
print("The number of Test requests: ",booking_requests_df_avant.shape[0]-booking_requests_df.shape[0])

The number of Test requests:  23472


In [1378]:
#Check if there are any negative values in "price_eur"
booking_requests_df = booking_requests_df[booking_requests_df["price_eur"]>0]
booking_requests_df.shape

(107594, 12)

In [1379]:
booking_requests_df["status"].unique()

<StringArray>
['fulfilled']
Length: 1, dtype: str

In [1380]:
booking_requests_df['rating'].unique()

array([5., 4., 3., 1., 2.])

# 4. Marketing_spend

In [1381]:
marketing_spend_df.head()

,month,city,side,channel,spend_eur
0,2025-01-01,Paris,demand,Google Ads,3920.0
1,2025-01-01,Paris,demand,Meta Ads,2880.0
2,2025-01-01,Paris,supply,Job boards,1620.0
3,2025-01-01,Paris,supply,Meta Ads,480.0
4,2025-01-01,Lyon,demand,Google Ads,2290.0


In [1382]:
marketing_spend_df.shape

(243, 5)

In [1383]:
marketing_spend_df = marketing_spend_df.drop_duplicates()

In [1384]:
marketing_spend_df.info()
# ==> No mising values neither duplicated rows

<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   month      243 non-null    str    
 1   city       243 non-null    str    
 2   side       243 non-null    str    
 3   channel    243 non-null    str    
 4   spend_eur  243 non-null    float64
dtypes: float64(1), str(4)
memory usage: 9.6 KB


In [1385]:
marketing_spend_df['city'].unique()

<StringArray>
['Paris', 'Lyon', 'Bordeaux', 'Lille']
Length: 4, dtype: str

In [1386]:
#Check if there are any negative values in "spend_eur"
marketing_spend_df = marketing_spend_df[marketing_spend_df["spend_eur"]>0]
marketing_spend_df.shape

(243, 5)

In [1387]:
marketing_spend_df['side'].unique()

<StringArray>
['demand', 'supply']
Length: 2, dtype: str